# Payments Data Analysis


Dataset created by Caixabank Tech for the 2024 AI Hackathon
Time Period: 2010s decade

1. Transaction Data (transactions_data.csv)
Detailed transaction records including amounts, timestamps, and merchant details
Covers transactions throughout the 2010s
Features transaction types, amounts, and merchant information
Perfect for analyzing spending patterns and building fraud detection models
2. Card Information (cards_dat.csv)
Credit and debit card details
Includes card limits, types, and activation dates
Links to customer accounts via card_id
Essential for understanding customer financial profiles
3. Merchant Category Codes (mcc_codes.json)
Standard classification codes for business types
Enables transaction categorization and spending analysis
Industry-standard MCC codes with descriptions
4. Fraud Labels (train_fraud_labels.json)
Binary classification labels for transactions
Indicates fraudulent vs. legitimate transactions
Ideal for training supervised fraud detection models
5. User Data (users_data)
Demographic information about customers
Account-related details
Enables customer segmentation and personalized analysis

# Installing packages and workspace preparation

In [0]:
%pip install kagglehub

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")

print("Path to dataset files:", path)

In [0]:
import pandas as pd

transactions = pd.read_csv(f"{path}/transactions_data.csv")
cards = pd.read_csv(f"{path}/cards_data.csv")
users = pd.read_csv(f"{path}/users_data.csv")

In [0]:
transactions_2010 = transactions[transactions['date'].str.startswith('2010')]
transactions_2010_spark = spark.createDataFrame(transactions_2010)
transactions_2010_spark.write.saveAsTable("transactions_2010")

In [0]:
# Created Spark Dataset
cards_spark = spark.createDataFrame(cards)
users_spark = spark.createDataFrame(users)
# Abbiamo caricato dataset in catalog
cards_spark.write.saveAsTable("cards")
users_spark.write.saveAsTable("users")

## EDA

In [0]:
%sql
---- Explore dataset
SELECT * FROM transactions_2010

In [0]:
%sql
--- Aggregate by year and month total amount of transactions
SELECT year(date) as year,
       month(date) as month,
       sum(cast(substring(amount, 2) AS decimal(10,2))) as total ---- $143.72  Opera: 1. substring 143.72 2. cast to decimal 3. sum 
FROM transactions_2010
GROUP BY year(date),month(date);

In [0]:
%sql 
CREATE TABLE view_client_trx AS
SELECT
    client_id,
    COUNT(DISTINCT card_id) unique_cards, 
    COUNT(*) trxs
FROM workspace.default.transactions_2010
GROUP BY client_id
ORDER BY trxs DESC

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--- How many cards and transaction on average clients have
-----> Reference data, fact data (transazionali)
WITH client_transactions (
  SELECT 
    client_id,
    COUNT(DISTINCT card_id) unique_cards, 
    COUNT(*) trxs
  FROM workspace.default.transactions_2010
  GROUP BY client_id
  ORDER BY trxs DESC
) 
SELECT 
avg(unique_cards),
avg(trxs),
median(trxs),
max(trxs),
min(trxs)
FROM client_transactions